In [ ]:
from googleapiclient.discovery import build
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

# La tua API Key YouTube
API_KEY = 'LA_TUA_API_KEY'

# Inizializza il client YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Canali target (metti i loro ID corretti)
channels = {
    "TechDale": "UCBJycsmduvYEL83R_U4JriQ",
    "Marques Brownlee": "UCBJycsmduvYEL83R_U4JriQ",
    "AppleInsider": "UCNf8CW9ir76u6pW9Vj4tDZQ"
}

query = "Apple vision pro review"

# Inizializza VADER
analyzer = SentimentIntensityAnalyzer()

In [ ]:
def search_videos_by_channel(youtube, channel_id, query, max_results=10):
    response = youtube.search().list(
        channelId=channel_id,
        q=query,
        part="id,snippet",
        maxResults=max_results,
        type="video",
        order="date"
    ).execute()

    video_ids = []
    for item in response['items']:
        video_ids.append({
            "videoId": item['id']['videoId'],
            "title": item['snippet']['title'],
            "publishedAt": item['snippet']['publishedAt']
        })
    return video_ids

In [ ]:
def get_comments(youtube, video_id, max_comments=100):
    comments = []
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )
    response = request.execute()

    while response:
        for item in response['items']:
            comment = item['snippet']['topLevelComment']['snippet']['textDisplay']
            comments.append(comment)
            if len(comments) >= max_comments:
                break
        if 'nextPageToken' in response and len(comments) < max_comments:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=response['nextPageToken'],
                maxResults=100,
                textFormat="plainText"
            )
            response = request.execute()
        else:
            break
    return comments

In [ ]:
all_comments = []

for channel_name, channel_id in channels.items():
    print(f"Cercando video nel canale {channel_name}...")
    videos = search_videos_by_channel(youtube, channel_id, query, max_results=5)
    for video in videos:
        print(f"Scaricando commenti per video: {video['title']}")
        comments = get_comments(youtube, video['videoId'], max_comments=50)
        for comment in comments:
            score = analyzer.polarity_scores(comment)
            all_comments.append({
                "channel": channel_name,
                "video_id": video['videoId'],
                "video_title": video['title'],
                "publishedAt": video['publishedAt'],
                "comment": comment,
                "neg": score['neg'],
                "neu": score['neu'],
                "pos": score['pos'],
                "compound": score['compound']
            })

In [ ]:
df = pd.DataFrame(all_comments)

print("Sentiment medio (compound) per canale:")
print(df.groupby('channel')['compound'].mean())

In [ ]:
df.to_csv("apple_vision_pro_comments_sentiment.csv", index=False)